In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import glob
import xarray as xr
import numpy as np
from distributed import Client, LocalCluster
import dask
import pickle
import os
from scipy.stats import linregress
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.stats import linregress
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
import joblib # For saving our models
import time
import emcee
from matplotlib.ticker import FuncFormatter
plt.rcParams['text.usetex'] = True
import corner # The library for creating corner plots

# Start Dask client

In [2]:
dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 

In [3]:
iparallel = True
if iparallel:
    cluster = LocalCluster(n_workers=25,threads_per_worker=1,memory_limit='8.5GB')#,dashboard_address=':8787')
    client = Client(cluster)

In [4]:
cluster

LocalCluster(473477f6, 'tcp://127.0.0.1:42383', workers=25, threads=25, memory=197.91 GiB)

# Work with full 3D files

In [8]:
def process_single_file(file):
    ds = xr.open_dataset(file)

    # Step 2: Crop data to start from 2010
    if ds.time.dt.year.min() < 2010:
        ds = ds.sel(time=slice('2010-01-01', None))
    
    lat = ds['lat'].values
    lon = ds['lon'].values
    icc = ds['icc2'].values # ice cloud fraction (-)
    ttop = ds['ttop2'].values # tempeature at top of liquid cloud layer (K)
    lcc = ds['lcc2'].values # liquid cloud fraction (-)
    lwp = ds['lwp2'].values # liquid water path (kg/m^2)
    iwp = ds['iwp2'].values # ice water path (kg/m^2)
    cdr = ds['cdr'].values # droplet effective radius at top of liquid cloud layer (m)
    cdnc = ds['cdnc'].values # droplet number concentration at top of liquid cloud layer (/m^3)
    cod = ds['cod'].values # cloud optical depth (-)
    ocnfrac = ds['OCNFRAC'].values # fraction of surface area coverd by ocean (-)
    area = ds['area'].values # physics grid area (??)
    omega_500 = ds['OMEGA500'].values # Vertical velocity at 500 mbar pressure surfac (Pa/s)
    omega_700 = ds['OMEGA700'].values # Vertical velocity at 700 mbar pressure surfac (Pa/s)
    th7001000 = ds['TH7001000'].values # Theta difference 700 mb - 1000 mb (K); Also known as lower tropospheric stability, or LTS
    omega_500 = omega_500*1.e-2*(3600.*24.) # Convert from Pa/s to hPa/day
    omega_700 = omega_700*1.e-2*(3600.*24.) # Convert from Pa/s to hPa/day
    
    fsnt = ds['FSNT'].values # Net solar flux at top of model (W/m^2)
    fsnt_d1 = ds['FSNT_d1'].values # Net solar flux at top of model WITHOUT aerosols(W/m^2)
    flnt = ds['FLNT'].values # Net longwave flux at top of model (W/m^2)
    flnt_d1 = ds['FLNT_d1'].values # Net longwave flux at top of model WITHOUT aerosols (W/m^2)
    flntc = ds['FLNTC'].values # Clearsky net longwave flux at top of model (W/m^2)
    flntc_d1 = ds['FLNTC_d1'].values # Clearsky net longwave flux at top of model WITHOUT aerosols (W/m^2)
    fsntc = ds['FSNTC'].values # Clearsky net solar flux at top of model (W/m^2)
    fsntc_d1 = ds['FSNTC_d1'].values # Clearsky net solar flux at top of model WITHOUT aerosols (W/m^2)
    
    flut = ds['FLUT'].values # Upwelling longwave flux at top of model
    flutc = ds['FLUTC'].values # Clearsky upwelling longwave flux at top of model
    fsutoa = ds['FSUTOA'].values # Upwelling solar flux at top of atmosphere
    fsutoac = ds['FSUTOAC'].values # Clearsky upwelling solar flux at top of atmosphere
    
    solin = ds['SOLIN'].values # Solar Insolation
    
    prect = ds['PRECT'].values # Total (convective and large-scale) precipitation rate (liq + ice) (m/s)
    precl = ds['PRECL'].values # Large-scale (stable) precipitation rate (liq + ice) (m/s)
    precz = ds['PRECZ'].values # Total precipitation from ZM convection (m/s)
    precc = ds['PRECC'].values # Convective precipitation rate (liq + ice) (m/s)
    prect = prect*1.e3*3600. # Convert from m/s to mm/hr
    precl = precl*1.e3*3600. # Convert from m/s to mm/hr
    precz = precz*1.e3*3600. # Convert from m/s to mm/hr
    precc = precc*1.e3*3600. # Convert from m/s to mm/hr
    autoconv = ds['autoconv'].values # Vertically integrated autoconversion rate (kg/m^2/s)
    accretn = ds['accretn'].values # Vertically integrated accretion rate (kg/m^2/s)
    
    ds.close()


    out_dict = {'lat':lat,\
                'lon':lon,\
                'ttop':ttop,\
                'lcc':lcc,\
                'lwp':lwp,\
                'cdnc':cdnc,\
                'cdr':cdr,\
                'iwp':iwp,\
                'icc':icc,\
                'cod':cod,\
                'area':area,\
                'omega_500':omega_500,\
                'omega_700':omega_700,\
                'th7001000':th7001000,\
                'ocnfrac':ocnfrac,\
                'fsnt':fsnt,\
                'fsnt_d1':fsnt_d1,\
                'flnt':flnt,\
                'flnt_d1':flnt_d1,\
                'fsntc':fsntc,\
                'fsntc_d1':fsntc_d1,\
                'flntc':flntc,\
                'flntc_d1':flntc_d1,\
                'prect':prect,\
                'precl':precl,\
                'precz':precz,\
                'precc':precc,\
                'autoconv':autoconv,\
                'accretn':accretn,\
                'flut':flut,\
                'flutc':flutc,\
                'fsutoa':fsutoa,\
                'fsutoac':fsutoac,\
                'solin':solin,\
               }

    return out_dict

def concat_futures(gathered_futures_list):
    
    # Exclude lat/lon from time-stacked variables
    vars_to_concat = [k for k in gathered_futures_list[0].keys() if k not in ("lat", "lon","area")]
    
    stacked_dict = {}
    for var in vars_to_concat:
        arrays = [d[var] for d in gathered_futures_list]  # each one is (time_i, ncol)
        stacked_dict[var] = np.concatenate(arrays, axis=0)  # final shape: (total_time, ncol)
    
    # Assume lat/lon are consistent across files; take from first
    lat = gathered_futures_list[0]['lat']
    lon = gathered_futures_list[0]['lon']
    area = gathered_futures_list[0]['area']
    
    stacked_dict['lat'] = lat
    stacked_dict['lon'] = lon
    stacked_dict['area'] = area

    return stacked_dict

def compute_cloud_fracs(stacked_dict):
    """
    Computes the warm cloud fraction, the stratiform fraction, and the warm stratiform fraction.
    """
    lcc = stacked_dict['lcc']
    warm_mask = stacked_dict['warm_mask']
    strat_mask = stacked_dict['strat_mask_ms']
    
    warm_frac = warm_mask.mean(axis=0)
    strat_frac = strat_mask.mean(axis=0)
    
    stacked_dict['warm_frac'] = warm_frac
    stacked_dict['strat_frac'] = strat_frac
    
    stacked_dict['warm_masked'] = np.ma.masked_where(warm_frac < 0.1, warm_frac)
    stacked_dict['strat_masked'] = np.ma.masked_where(strat_frac < 0.1, strat_frac)
    
    # Define the stratiform fraction threshold
    strat_frac_thresh = 0.3
    
    # Boolean mask for valid stratiform-prone grid cells
    strat_frac_mask = (strat_frac > strat_frac_thresh)
    
    # Compute warm cloud frequency at each column
    warm_occurrence = warm_mask.mean(axis=0)  # shape (ncol,)
    
    # Apply stratiform constraint
    warm_strat_frac = np.where(strat_frac_mask, warm_occurrence, np.nan)
    
    # Save to output dictionary
    stacked_dict['warm_strat_frac'] = warm_strat_frac

    return stacked_dict

In [9]:
base_path = "/glade/derecho/scratch/nmahfouz/msp1/"
cases_pd = sorted(glob.glob(f"{base_path}msp1_pd_*"))
num_cases_pd = len(cases_pd)
print('# of PD cases:',num_cases_pd)
cases_pi = sorted(glob.glob(f"{base_path}msp1_pi_*"))
num_cases_pi = len(cases_pi)
print('# of PI cases:',num_cases_pi)
case_strs = []
case_ints = []
for ii in range(num_cases_pd):
    dum_str = cases_pd[ii].split('/')[-1].split('_')[-1]
    case_strs.append(dum_str)
    case_ints.append(int(dum_str))
print(case_strs)
print(case_ints)
save_path = "/glade/u/home/mckenna/scratch/ppe_processed_files/"

# of PD cases: 54
# of PI cases: 54
['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53']
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]


In [10]:
params_df = pd.read_csv('/glade/u/home/mckenna/work/ppe_pre/e3sm_lhs_samples.csv')
df_to_add_begin = pd.DataFrame([{'Sample #': 0, 'Accretion Factor': 1, 'Autoconversion Factor': 1}])
df_to_add_end = pd.DataFrame([
    {'Sample #': 51, 'Accretion Factor': 0, 'Autoconversion Factor': 1},
    {'Sample #': 52, 'Accretion Factor': 1, 'Autoconversion Factor': 0},
    {'Sample #': 53, 'Accretion Factor': 0, 'Autoconversion Factor': 0}
])
params_df = pd.concat([df_to_add_begin, params_df, df_to_add_end], ignore_index=True).set_index('Sample #')
accr_facs = []
auto_facs = []
sample_nums = []
for index, row in params_df.iterrows():
    # The 'index' is the 'Sample #'
    sample_num = float(index)
    sample_nums.append(sample_num)
    # Access columns from the 'row' object (which is a Pandas Series)
    acc_factor = float(row['Accretion Factor'])
    auto_factor = float(row['Autoconversion Factor'])
    accr_facs.append(acc_factor)
    auto_facs.append(auto_factor)

In [11]:
def driver_func(case_str,auto_fac,accr_fac):
#for ii in range(num_cases_pd):
    #print('Processing case:',cases_pd[ii],'; % done:',(ii+1)/num_cases_pd*100.)

    #pd_path = f"/glade/derecho/scratch/nmahfouz/msp1/msp1_pd_{case_strs[ii]}/run/"
    #pi_path = f"/glade/derecho/scratch/nmahfouz/msp1/msp1_pi_{case_strs[ii]}/run/"
    pd_path = f"/glade/derecho/scratch/nmahfouz/msp1/msp1_pd_{case_str}/run/"
    pi_path = f"/glade/derecho/scratch/nmahfouz/msp1/msp1_pi_{case_str}/run/"
    pd_files = sorted(glob.glob(pd_path+'*.h1.*.nc'))[6:]
    pi_files = sorted(glob.glob(pi_path+'*.h1.*.nc'))[6:]
    num_pd_files = len(pd_files)
    num_pi_files = len(pi_files)

    #----------------------------------
    # Loop through files and get variables
    #----------------------------------  

    #-----------------
    # PD
    #-----------------
    futures = []
    for jj in range(num_pd_files):
        future = client.submit(process_single_file, pd_files[jj])
        futures.append(future)
    gathered_futures_list = client.gather(futures)
    stacked_pd_dict = concat_futures(gathered_futures_list)

    assert all(np.allclose(d['area'], gathered_futures_list[0]['area']) for d in gathered_futures_list), "Mismatch in area across files"
    #-----------------
    # PI
    #-----------------
    futures = []
    for jj in range(num_pi_files):
        future = client.submit(process_single_file, pi_files[jj])
        futures.append(future)
    gathered_futures_list = client.gather(futures)
    stacked_pi_dict = concat_futures(gathered_futures_list)

    #----------------------------------
    # Write to dictionary
    #----------------------------------  
    # Create combined dictionary
    combined_dict = {
        'pD': stacked_pd_dict,
        'pI': stacked_pi_dict
    }
    #combined_dict['auto_fac'] = auto_facs[ii]
    #combined_dict['accr_fac'] = accr_facs[ii]
    combined_dict['auto_fac'] = auto_fac
    combined_dict['accr_fac'] = accr_fac
    
    # Construct prefix
    #out_prefix = f"{cases[ii]}"
    out_prefix = f"{case_str}"
    print(out_prefix)

    # Save as pickle
    with open(f"{save_path}/msp1_{out_prefix}.pkl", "wb") as f:
        pickle.dump(combined_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    #print(aaaaaa)
    return out_prefix


In [12]:
#======================================================================
#======================================================================
# Loop through cases and submit futures
#======================================================================
#======================================================================
for ii in range(num_cases_pd):
    print('Processing case:',cases_pd[ii],'; % done:',(ii+1)/num_cases_pd*100.)
    out_prefix = driver_func(case_strs[ii],auto_facs[ii],accr_facs[ii])

Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_00 ; % done: 1.8518518518518516
00
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_01 ; % done: 3.7037037037037033
01
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_02 ; % done: 5.555555555555555
02
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_03 ; % done: 7.4074074074074066
03
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_04 ; % done: 9.25925925925926
04
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_05 ; % done: 11.11111111111111
05
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_06 ; % done: 12.962962962962962
06
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_07 ; % done: 14.814814814814813
07
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_08 ; % done: 16.666666666666664
08
Processing case: /glade/derecho/scratch/nmahfouz/msp1/msp1_pd_09 ; % done: 18.51851851851852
09
Processing case: /glade/derecho/scr

In [ ]:
#======================================================================
# Function to process cases and write dictionary pickle file
#======================================================================
def driver_func(auto_prefix,accr_prefix,evap_prefix):
#for ii in range(num_cases_pd):
#    print('Processing case:',cases_pd[ii],'; % done:',(ii+1)/num_cases_pd*100.)

    # There should be 37 files for PD/PI cases
    #pd_path = f"/pscratch/sd/m/mckenna/e3sm/lcrc/globalscratch/ac.ngmahfouz/v3nm/v3n2_{auto_pre[ii]}_{accr_pre[ii]}_{evap_pre[ii]}_pD/run/"
    #pi_path = f"/pscratch/sd/m/mckenna/e3sm/lcrc/globalscratch/ac.ngmahfouz/v3nm/v3n2_{auto_pre[ii]}_{accr_pre[ii]}_{evap_pre[ii]}_pI/run/"
    pd_path = f"/pscratch/sd/m/mckenna/e3sm/lcrc/globalscratch/ac.ngmahfouz/v3nm/v3n2_{auto_prefix}_{accr_prefix}_{evap_prefix}_pD/run/"
    pi_path = f"/pscratch/sd/m/mckenna/e3sm/lcrc/globalscratch/ac.ngmahfouz/v3nm/v3n2_{auto_prefix}_{accr_prefix}_{evap_prefix}_pI/run/"
    pd_files = sorted(glob.glob(pd_path+'*.h1.*.nc'))
    pi_files = sorted(glob.glob(pi_path+'*.h1.*.nc'))
    pd_files = pd_files[int(len(pd_files)/2):]
    pi_files = pi_files[int(len(pi_files)/2):]
    num_pd_files = len(pd_files)
    num_pi_files = len(pi_files)
    #print('# of PD files:',num_pd_files)
    #print('# of PI files:',num_pi_files)


    #----------------------------------
    # Loop through files and get variables
    #----------------------------------  

    #-----------------
    # PD
    #-----------------
    futures = []
    for jj in range(num_pd_files):
        future = client.submit(process_single_file, pd_files[jj])
        futures.append(future)
    gathered_futures_list = client.gather(futures)
    stacked_pd_dict = concat_futures(gathered_futures_list)

    assert all(np.allclose(d['area'], gathered_futures_list[0]['area']) for d in gathered_futures_list), "Mismatch in area across files"
    #-----------------
    # PI
    #-----------------
    futures = []
    for jj in range(num_pi_files):
        future = client.submit(process_single_file, pi_files[jj])
        futures.append(future)
    gathered_futures_list = client.gather(futures)
    stacked_pi_dict = concat_futures(gathered_futures_list)

    #stacked_pd_dict = compute_cloud_fracs(stacked_pd_dict)
    #stacked_pi_dict = compute_cloud_fracs(stacked_pi_dict)


    #----------------------------------
    # Write to dictionary
    #----------------------------------  
    # Create combined dictionary
    combined_dict = {
        'pD': stacked_pd_dict,
        'pI': stacked_pi_dict
    }
    
    # Construct prefix
    out_prefix = f"{auto_prefix}_{accr_prefix}_{evap_prefix}"
    
    # Save as pickle
    with open(f"/pscratch/sd/m/mckenna/e3sm/pickled_dicts/v3n2_{out_prefix}.pkl", "wb") as f:
        pickle.dump(combined_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    return out_prefix
